In [ ]:
import numpy as np
import os
import pandas as pd
from cmcrameri import cm
import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
import csv
import json
import os
import pandas as pd
import time
import re
from cmcrameri import cm
import xarray as xr

plt.style.use('./libs/notebookstyle_aps.mplstyle')

%load_ext autoreload
%autoreload 2

## Data sets

In [ ]:
# options for opening datasets
kwargs = {'decode_times': False, 'chunks': 'auto'}
basepath = '/work/bd1179/palm/JOBS_Lena/'


ds_3d = {}
ts = {}

# first series of data sets (numbering starting with 3)
datasets = [
    ('a1', 3),
    ('a2', 4),
    ('a3', 5),
    ('a4', 6),
    ('a5', 7),
    ('a6', 8),
]

for key, index in datasets:
    ds_3d[key] = xr.open_mfdataset(f'{basepath}qml2a/OUTPUT/qml2a_3d.00{index}.nc', **kwargs)
    ts[key] = xr.open_mfdataset(f'{basepath}qml2a/OUTPUT/qml2a_ts.00{index}.nc', **kwargs)

# additional series with different prescribed surface heat flux
labels = ['b', 'b', 'b', 'b','b', 'b',
          'c', 'c', 'c', 'c', 'c', 'c',
         'd','d','d', 'd','d','d']
indices = [1, 2, 3, 4, 5, 6,
           1, 2, 3, 4, 5, 6,
           0, 1, 2, 3, 4, 5]

for label, index in zip(labels, indices):
    key = f"{label}{int(index + (1 if label == 'd' else 0))}"
    path = f'{basepath}qml2{label}/OUTPUT/qml2{label}'
    ds_3d[key] = xr.open_mfdataset(f'{path}_3d.00{index}.nc', **kwargs)
    ts[key] = xr.open_mfdataset(f'{path}_ts.00{index}.nc', **kwargs)

series = [ 'a1','a2','a3','a4','a5','a6','b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'c1', 'c2','c3','c4','c5','c6', 'd1', 'd2','d3','d4','d5','d6']

In [ ]:
# plot max velocity u, different bands correspond to different prescribed geostrophic wind
for ser in series:
    if np.mod(int(ser[-1])-1,6) == 0:
        plt.figure()
    ts[ser].umax.plot(label = ser)
    plt.legend()

In [ ]:
for ser in series:
    if np.mod(int(ser[-1])-1,6) == 0:
        plt.figure()
    ds_3d[ser].isel(time = 10, x = 20).u.plot(label = ser, alpha = 0.3, bins = 50)
    plt.legend()
    

In [ ]:
# potential temperature
ds_3d['b4'].isel(x = 10, y = 200, time = 1).theta.plot()
ds_3d['b2'].isel(x = 10, y = 200, time = 1).theta.plot()
ds_3d['b6'].isel(x = 10, y = 200, time = 1).theta.plot()
ds_3d['b1'].isel(x = 10, y = 200, time = 1).theta.plot()
ds_3d['b3'].isel(x = 10, y = 200, time = 1).theta.plot()
ds_3d['b5'].isel(x = 10, y = 200, time = 1).theta.plot()

In [ ]:
# height of the atmospheric boundary layer top
for i in series:
    ts[i].abl.plot(label = i)
plt.fill_betweenx(x1 = -100,x2 = 2700, y = [900,1400], color = 'lightgrey')
plt.legend(loc = [1.02,0])

In [ ]:
for j,i in enumerate(series):
    plt.plot(ts[i]['theta(z_mo)'], label = i )
plt.axhline(302.3)
plt.axhline(301.8)
plt.legend(loc = [1.01,0])

In [ ]:
# save abl heights at the time steps from the 3D files as txt
! mkdir abl_heights
i = 0
for l in series:
    dsx = ds_3d[l]
    tsx = ts[l]
    tsx.abl.interp(time = dsx.time.values).to_dataframe().to_csv('./abl_heights/ABL_%s.txt'%l)
    tsx.abl.plot()
    tsx.abl.interp(time = dsx.time.values).to_dataframe().abl.plot(color = 'k')

In [ ]:
def make_same_indices(dsc):
    # loads the xarray dataset on palm's grid and casts onto indices for each cell, ie state var grid coordinate goes together with the corresponding u,v,w grid coordinate
    # returns a dataset
    dsnew = xr.Dataset()
    for var in dsc.variables:
        if var not in ['x','xu','y','yv','zu_3d','zw_3d','time']:
            if len(dsc[var].coords)==4:
                dsnew[var] = xr.DataArray(dsc[var].values, dims = ['time','Z','Y','X'])
            else:
                dsnew[var] = xr.DataArray(dsc[var].values, dims = ['time','Z'])
    dsnew['time'] = dsc.time.values
    return dsnew

In [ ]:
### save the coarse grained series with the relative height coordinates as txt
for label in series:
    pathx = './abl_heights/ABL_%s.txt'%label
    print(pathx)
    abl_ts = pd.read_csv(pathx, index_col = 0).dropna()
    dsnew = xr.Dataset()
    for time_index in np.arange(1,277):
        print(time_index)
        dsc = xr.open_mfdataset("/work/bd1179/b309252/qml2_cg/qml2%s_%s_cg200_10z_interpna_000%03d.nc"%(label[0],int(label[1]) + 2*(label[0] == 'a') - 1*(label[0] == 'd'),time_index))
        dsc = dsc.sel(zw_3d = slice(0,1350), zu_3d = slice(0,1450))
        dsc['zrel_u'] = xr.DataArray([dsc.zu_3d.values/abl_ts.iloc[time_index-1].abl], dims = ['time','Z'])
        dsc['zrel_w'] = xr.DataArray([dsc.zw_3d.values/abl_ts.iloc[time_index-1].abl], dims = ['time','Z'])
        dssame = make_same_indices(dsc)
        dsnew = dsnew.merge(dssame)
    dsnew = dsnew.where(dsnew.zrel_u < 1, drop = True)
    dsnew = dsnew.where(dsnew.zrel_w < 1, drop = True)
    dsnew.to_dataframe().to_csv('./cg_data/QML2_%s_cg200_10z.txt'%label)

## check the series

In [ ]:
df_all = pd.DataFrame()
for label in series:
    data_path = '/work/bd1179/b309252/QML2/QML2_%s_cg200_10z.txt'%label ### replace with ./cg_data/ or what else the actual path is
    df = pd.read_csv(data_path).dropna()
    df_all = pd.concat([df_all,df])
    (df.theta_w_SGS ).hist(bins = 200, alpha = 0.4)

In [ ]:
df_all.theta_w_SGS.hist(bins = 200, alpha = 0.4)